# MultiLeRobotDataset 教程

> 合并多个 `TransformedLeRobotDataset`，并在训练时按 `dataset_weights` 做加权采样。

## ⚠️ 重要：这不是 LeRobot 官方实现

本教程讲的是 **TBot / my_tbot fork** 里的扩展版，**不是** HuggingFace 官方 `lerobot` 包里的同名类。

| | 官方 LeRobot | TBot (my_tbot) — **本教程** |
|--|-------------|------------------------------|
| 文件 | `lerobot_dataset.py` | `transformed_dataset.py` |
| 构造方式 | `MultiLeRobotDataset(repo_ids=[...])` 内部建子 Dataset | `MultiLeRobotDataset(transformed_datasets, dataset_weights=...)` |
| 子数据集 | 原始 `LeRobotDataset` | 已 transform 的 `TransformedLeRobotDataset` |
| `dataset_weights` | ❌ 无 | ✅ 有，配合 `MultiLeRobotWeightedSampler` |
| YAML weight rules | ❌ 无 | ✅ `factory.make_dataset` + `weight_rules_path` |

**my_vla 环境**：默认 `sys.path` 指向 `/vla/my_vla/src/lerobot`（或 pip `lerobot 0.4.4`），**没有** `transformed_dataset.py`，直接 `import` 会报错。

**正确做法**（本 notebook 已包含）：
```python
sys.path.insert(0, "/vla/my_tbot/src")  # 必须放在最前，覆盖 my_vla 的 lerobot
```

- **第 3–8 节核心 demo**：在 `myvla` 环境下 **可以运行**（只要加了上面的 path）。
- **第 9 节 `make_dataset()`**：需要 `omegaconf`，`myvla` 目前未安装；完整预训练请用 **`tbot_sa1`** 环境。

**源码位置**（my_tbot）：
- `lerobot/datasets/transformed_dataset.py` → `MultiLeRobotDataset`
- `lerobot/datasets/sampler.py` → `MultiLeRobotWeightedSampler`
- `lerobot/datasets/factory.py` → `make_dataset()` 自动构建
- `lerobot/scripts/lerobot_train.py` → 训练时自动选择 Sampler

---

## 1. 它解决什么问题？

预训练/多源训练时，你往往有多个 robot / 多个 repo：

```
repo A: adjust_bottle  (10548 frames)
repo B: test           (436 frames)
repo C: robotwin/...   (...)
```

你需要：
1. **统一索引空间**：DataLoader 只看到一个 `Dataset`，`len()` = 所有子数据集帧数之和。
2. **控制采样比例**：小数据集不应被大数据集"淹没"，或按业务预算（如 47% interndata + 7% robotwin）采样。

`MultiLeRobotDataset` 负责 (1)，`dataset_weights` + `MultiLeRobotWeightedSampler` 负责 (2)。

---

## 2. 核心架构

```
┌─────────────────────────────────────────────────────────────┐
│                    MultiLeRobotDataset                        │
│  len = sum(num_frames_i)                                    │
│  __getitem__(global_idx) → 路由到子数据集 local_idx          │
│  dataset_weights: Optional[Tensor]  (归一化后的采样概率)     │
└───────────────┬─────────────────────┬───────────────────────┘
                │                     │
    ┌───────────▼──────────┐  ┌───────▼──────────┐
    │ TransformedLeRobot   │  │ TransformedLeRobot│  ...
    │ Dataset (repo A)     │  │ Dataset (repo B)  │
    │ num_frames = 10548   │  │ num_frames = 436  │
    └──────────────────────┘  └───────────────────┘

训练 DataLoader:
  dataset_weights is None  → shuffle=True，按帧均匀遍历
  dataset_weights 有值     → MultiLeRobotWeightedSampler，按权重抽 dataset 再均匀抽帧
```

**关键设计**：`MultiLeRobotDataset` 本身仍是 **concat 索引**（global index 连续拼接），
加权采样由 **Sampler** 完成，而不是改 `__getitem__` 逻辑。

## 3. 环境准备

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import torch
from torch.utils.data import DataLoader

# ★ 必须用 my_tbot 源码，不能用 my_vla / pip 自带的官方 lerobot
TBOT_SRC = Path("/vla/my_tbot/src")
if str(TBOT_SRC) in sys.path:
    sys.path.remove(str(TBOT_SRC))
sys.path.insert(0, str(TBOT_SRC))  # insert(0) 确保优先于 my_vla 的 lerobot

import lerobot
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.datasets.transformed_dataset import (
    MultiLeRobotDataset,
    TransformedLeRobotDataset,
)
from lerobot.datasets.sampler import MultiLeRobotWeightedSampler

assert hasattr(lerobot.datasets, "transformed_dataset") or Path(TBOT_SRC / "lerobot/datasets/transformed_dataset.py").exists()
print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("lerobot 实际加载自:", Path(lerobot.__file__).parent)
print("是否为 TBot fork:", "my_tbot" in str(Path(lerobot.__file__).resolve()))

## 4. 索引路由机制（无需真实数据）

`MultiLeRobotDataset._locate_dataset(global_idx)` 把全局索引映射到 `(dataset_index, local_index)`：

| 子数据集 | 帧数 | global index 范围 |
|---------|------|------------------|
| ds[0]   | 1000 | [0, 999]         |
| ds[1]   | 800  | [1000, 1799]     |
| ds[2]   | 200  | [1800, 1999]     |

下面用 **Mock 子数据集** 演示路由逻辑（与真实 `TransformedLeRobotDataset` 行为一致）。

In [ ]:
class MockTransformedDataset:
    """最小 Mock：只实现 MultiLeRobotDataset 需要的接口。"""

    def __init__(self, repo_id: str, num_frames: int, num_episodes: int = 1):
        self.repo_id = repo_id
        self.num_frames = num_frames
        self.meta = type("Meta", (), {"total_episodes": num_episodes, "episodes": {
            "dataset_from_index": list(range(0, num_frames, max(1, num_frames // num_episodes))),
            "dataset_to_index": list(range(max(1, num_frames // num_episodes), num_frames + 1, max(1, num_frames // num_episodes))),
        }})()

    def __len__(self):
        return self.num_frames

    def __getitem__(self, idx):
        return {"repo_id": self.repo_id, "local_idx": idx, "action": torch.zeros(7)}


mock_a = MockTransformedDataset("repo_a", num_frames=1000, num_episodes=10)
mock_b = MockTransformedDataset("repo_b", num_frames=800, num_episodes=8)
mock_c = MockTransformedDataset("repo_c", num_frames=200, num_episodes=2)

multi_mock = MultiLeRobotDataset([mock_a, mock_b, mock_c])
print(multi_mock)
print("总帧数:", len(multi_mock))

# 边界测试
test_indices = [0, 999, 1000, 1799, 1800, 1999, -1]
for gi in test_indices:
    ds_idx, local_idx = multi_mock._locate_dataset(gi)
    sample = multi_mock[gi]
    print(f"global {gi:5d} → ds[{ds_idx}] local={local_idx:4d}  repo={sample['repo_id']}")

## 5. 实战：用本地 LeRobot 数据集构建 MultiLeRobotDataset

本 notebook 使用两个本地数据集：
- `/vla/.data/adjust_bottle` — 约 10548 帧
- `/vla/.data/test` — 约 436 帧

生产环境中，每个子数据集会先经过 `_build_single_dataset()` 包装成 `TransformedLeRobotDataset`（含 normalize、delta action 等 transform）。
这里为演示简洁，直接用 `TransformedLeRobotDataset.from_repo()`，transforms 传 `[]`（空 pipeline，不做额外变换）。

In [ ]:
REPO_A = "/vla/.data/adjust_bottle"
REPO_B = "/vla/.data/test"

ds_a = TransformedLeRobotDataset.from_repo(
    repo_id=REPO_A,
    download_videos=False,
    transforms=[],
)
ds_b = TransformedLeRobotDataset.from_repo(
    repo_id=REPO_B,
    download_videos=False,
    transforms=[],
)

print(f"A: {ds_a.repo_id}  frames={ds_a.num_frames}  episodes={ds_a.meta.total_episodes}")
print(f"B: {ds_b.repo_id}  frames={ds_b.num_frames}  episodes={ds_b.meta.total_episodes}")

transformed_datasets = [ds_a, ds_b]
multi_ds = MultiLeRobotDataset(transformed_datasets)
print(multi_ds)

## 6. `dataset_weights` 参数详解

```python
multi_ds = MultiLeRobotDataset(transformed_datasets, dataset_weights=dataset_weights)
```

| 参数 | 含义 |
|------|------|
| `transformed_datasets` | `Sequence[TransformedLeRobotDataset]`，每个元素对应一个 repo |
| `dataset_weights=None` | 不加权；训练时 `shuffle=True`，每个 epoch 遍历全部帧 |
| `dataset_weights=[w0, w1, ...]` | 与 datasets **等长**；内部自动归一化为概率；训练时用 `MultiLeRobotWeightedSampler` |

**注意**：权重只需 **相对大小**，`[0.9, 0.1]` 与 `[9, 1]` 等价。

**factory.py 中的写法**（多 repo 预训练）：

```python
dataset_weights = [
    repo_weights_map[ds.repo_id]
    for ds in transformed_datasets
] if repo_weights_map is not None else None

multi_ds = MultiLeRobotDataset(transformed_datasets, dataset_weights=dataset_weights)
```

其中 `repo_weights_map` 由 YAML `weight_rules_path` 经 `compute_repo_weights()` 计算。

In [ ]:
# 场景 1：不加权（concat + shuffle）
multi_uniform = MultiLeRobotDataset(transformed_datasets, dataset_weights=None)
print("不加权:", multi_uniform.dataset_weights)

# 场景 2：显式权重 — 让小数据集 test 占 30% 采样（尽管它只有 ~4% 的帧）
raw_weights = [0.7, 0.3]  # adjust_bottle : test
multi_weighted = MultiLeRobotDataset(transformed_datasets, dataset_weights=raw_weights)
print("原始权重:", raw_weights)
print("归一化后:", multi_weighted.dataset_weights.tolist())
print("子数据集帧数:", multi_weighted._lengths)

## 7. MultiLeRobotWeightedSampler：加权采样的实际行为

Sampler 每一步：
1. 按 `dataset_weights`  multinomial 选一个子数据集 `ds_idx`
2. 在该子数据集内 **均匀** 随机选一个 `local_idx`
3. 转成 global index 返回给 DataLoader

因此：**dataset 级别按权重，dataset 内部仍均匀**。

In [ ]:
N_SAMPLES = 20_000
g = torch.Generator().manual_seed(42)
sampler = MultiLeRobotWeightedSampler(
    dataset=multi_weighted,
    num_samples=N_SAMPLES,
    generator=g,
)

counts = Counter()
for global_idx in sampler:
    ds_idx, local_idx = multi_weighted._locate_dataset(global_idx)
    counts[ds_idx] += 1

names = [Path(ds.repo_id).name for ds in transformed_datasets]
print(f"模拟 {N_SAMPLES} 次采样:\n")
for i, name in enumerate(names):
    pct = counts[i] / N_SAMPLES * 100
    frame_pct = multi_weighted._lengths[i] / sum(multi_weighted._lengths) * 100
    target_pct = multi_weighted.dataset_weights[i].item() * 100
    print(f"  ds[{i}] {name:20s}  采样={pct:5.1f}%  目标权重={target_pct:5.1f}%  帧占比={frame_pct:5.1f}%")

对比：**不加权 shuffle** 时，每个子数据集的采样比例 ≈ 其帧数占比（test 约 4%）。
加权 `[0.7, 0.3]` 后，test 被提升到 30%。

In [ ]:
# 对比：均匀 concat 索引采样（模拟 shuffle 遍历 random subset）
uniform_counts = Counter()
g2 = torch.Generator().manual_seed(42)
for _ in range(N_SAMPLES):
    global_idx = torch.randint(len(multi_uniform), (1,), generator=g2).item()
    ds_idx, _ = multi_uniform._locate_dataset(global_idx)
    uniform_counts[ds_idx] += 1

print("均匀索引随机采样（近似 shuffle 行为）:")
for i, name in enumerate(names):
    pct = uniform_counts[i] / N_SAMPLES * 100
    print(f"  ds[{i}] {name:20s}  采样={pct:5.1f}%")

## 8. 与 DataLoader / 训练脚本对接

`lerobot_train.py` 中的逻辑：

```python
if hasattr(dataset, "dataset_weights") and dataset.dataset_weights is not None:
    shuffle = False
    sampler = MultiLeRobotWeightedSampler(dataset=dataset)
else:
    shuffle = True
    sampler = None

dataloader = DataLoader(dataset, batch_size=..., shuffle=shuffle, sampler=sampler, ...)
```

**规则**：`sampler` 与 `shuffle=True` 不能同时使用。

In [ ]:
BATCH_SIZE = 4

loader = DataLoader(
    multi_weighted,
    batch_size=BATCH_SIZE,
    shuffle=False,
    sampler=MultiLeRobotWeightedSampler(multi_weighted, generator=torch.Generator().manual_seed(0)),
    num_workers=0,
)

batch = next(iter(loader))
print("batch keys:", list(batch.keys())[:8], "...")
print("batch size:", batch["action"].shape[0] if "action" in batch else "N/A")

# 取 1 个样本看结构
sample = multi_weighted[0]
print("\n单样本 keys:", list(sample.keys()))

## 9. 从 `make_dataset()` 一键构建（生产路径）

实际预训练不会手写上面的代码，而是走 `factory.make_dataset(cfg)`：

1. 从 `repo_id_file` 读取多个 repo
2. 可选 `weight_rules_path` YAML → `compute_repo_weights()`
3. 每个 repo 调用 `_build_single_dataset()` → `TransformedLeRobotDataset`
4. 若 `len(repo_ids) > 1` → `MultiLeRobotDataset(..., dataset_weights=...)`

下面演示 **最小化配置**（需根据你的环境修改路径）。

In [ ]:
# 可选：演示 make_dataset 路径（需要 omegaconf + 完整 TrainPipelineConfig）
RUN_MAKE_DATASET = True  # 设为 False 可跳过

if RUN_MAKE_DATASET:
    import tempfile
    try:
        from lerobot.configs.train import TrainPipelineConfig
        from lerobot.datasets.factory import make_dataset
    except ImportError as e:
        print(f"跳过 make_dataset 演示（缺少依赖: {e}）")
    else:
        repo_list = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False)
        repo_list.write(REPO_A + "\n")
        repo_list.write(REPO_B + "\n")
        repo_list.close()

        cfg_dict = {
            "dataset": {
                "repo_id": "multidata_from_file",
                "repo_id_file": repo_list.name,
                "video_backend": "pyav",
                "dist_loading": False,
            },
            "policy": {"type": "act"},
            "output_dir": "/tmp/multi_ds_demo",
            "batch_size": 2,
        }
        cfg = TrainPipelineConfig.from_dict(cfg_dict)
        dataset, data_stats = make_dataset(cfg)

        print(type(dataset).__name__)
        print(dataset)
        print("dataset_weights:", getattr(dataset, "dataset_weights", None))
        print("data_stats keys:", list(data_stats.keys()))

## 10. Streaming 模式：MultiStreamingLeRobotDataset

当 `cfg.dataset.streaming=True` 时，factory 返回 `MultiStreamingLeRobotDataset`：

| | 非 Streaming | Streaming |
|--|-------------|-----------|
| 类型 | `MultiLeRobotDataset` (map-style) | `MultiStreamingLeRobotDataset` (iterable) |
| 索引 | 支持 `__getitem__(idx)` | **不支持** random index |
| 无权重 | concat 顺序 yield | 各 stream 顺序 yield |
| 有权重 | `MultiLeRobotWeightedSampler` | `__iter__` 内每步按权重选 stream |
| DataLoader | `num_workers` 任意 | 通常 `num_workers=1`, `shuffle=False` |

大数据集（100M+ 帧）优先用 Streaming 版本。

## 11. 常见问题 FAQ

### Q1: 传了 weights 但采样比例不对？
- 确认训练时是否走了 `MultiLeRobotWeightedSampler`（TBot_SA1 等自定义 policy 可能用自己的 `ResumableEpochSampler`）。
- 确认 `dataset_weights` 长度与子数据集数量一致。

### Q2: 小数据集被大数据集淹没？
- 不加权时，采样比例 ≈ 帧数比例。要给 small repo 更高曝光，必须设 `dataset_weights` 或 YAML weight rules。

### Q3: `dist_loading=True` 时每个 rank 看到的数据不同？
- 多卡时 repo 会按 rank 分配子集；每个 rank 上的 `MultiLeRobotDataset` 可能只含 **部分** repo。
- 权重在 rank 内子集上重新归一化。

### Q4: 两个 repo 的 sample dict keys 不一致？
- `MultiLeRobotDataset` 假设 transform 已对齐 keys；否则 DataLoader collate 会失败。
- 确保 `_build_single_dataset` 对 all repos 使用相同的 transform pipeline。

### Q5: 与 `lerobot_dataset.py` 里旧的 `MultiLeRobotDataset` 混淆？
- my_tbot 训练路径用的是 `transformed_dataset.MultiLeRobotDataset`（本教程讲的这个）。
- `lerobot_dataset.py` 中有同名 legacy 类，factory 导入的是 transformed 版本。

## 12. 速查 Cheat Sheet

```python
from lerobot.datasets.transformed_dataset import (
    TransformedLeRobotDataset,
    MultiLeRobotDataset,
)
from lerobot.datasets.sampler import MultiLeRobotWeightedSampler

# 1) 构建子数据集
ds_list = [
    TransformedLeRobotDataset.from_repo("/path/to/repo_a", transforms=...),
    TransformedLeRobotDataset.from_repo("/path/to/repo_b", transforms=...),
]

# 2) 合并
weights = [0.7, 0.3]  # 或 None
multi_ds = MultiLeRobotDataset(ds_list, dataset_weights=weights)

# 3) DataLoader
if multi_ds.dataset_weights is not None:
    loader = DataLoader(
        multi_ds,
        batch_size=32,
        shuffle=False,
        sampler=MultiLeRobotWeightedSampler(multi_ds),
    )
else:
    loader = DataLoader(multi_ds, batch_size=32, shuffle=True)

# 4) 生产环境
# dataset, stats = make_dataset(cfg)  # cfg.dataset.repo_id_file + weight_rules_path
```